# finverse — full toolkit demo

This notebook walks through the complete finverse toolkit.
All cells run without API keys except the FRED section.

```bash
pip install finverse
```

In [ ]:
from finverse import pull, DCF, sensitivity, scenarios
from finverse.ml import forecast, garch, factor, anomaly
from finverse.credit import merton, altman
from finverse.risk import monte_carlo, evt, kelly
from finverse.portfolio import hrp
from finverse.audit import earnings_quality, benford
from finverse.valuation import real_options, apv
from finverse.macro import nelson_siegel
print('finverse ready')

## 1. Pull data

In [ ]:
data = pull.ticker('AAPL')
data.summary()

## 2. DCF model

In [ ]:
model = DCF(data)
model.set(wacc=0.095, terminal_growth=0.025)
model.run().summary()

In [ ]:
sensitivity(model, rows='wacc', cols='terminal_growth')

## 3. ML forecasting

In [ ]:
fc = forecast.revenue(data, n_years=5)
fc.summary()

In [ ]:
vol = garch.fit(data, model_type='GJR-GARCH')
vol.summary()

## 4. Credit analysis

In [ ]:
merton.analyze(data, garch_vol=vol.current_vol).summary()

In [ ]:
altman.analyze(data).summary()

## 5. Tail risk + Kelly sizing

In [ ]:
tail = evt.analyze(data)
tail.summary()

In [ ]:
sizing = kelly.from_distribution(data)
sizing.summary()
sizing.simulate(n_periods=252).plot(title='Kelly wealth paths', figsize=(10,4))

## 6. Monte Carlo

In [ ]:
mc = monte_carlo.simulate(model, n_simulations=5000)
mc.summary()
mc.plot()

## 7. Portfolio optimization

In [ ]:
tickers = ['AAPL', 'MSFT', 'GOOGL', 'JPM', 'XOM']
data_list = [pull.ticker(t) for t in tickers]
hrp.optimize(data_list).summary()

## 8. Real options

In [ ]:
real_options.expand(project_value=500, expansion_cost=200, sigma=0.30, time_to_expiry=3).summary()

## 9. Yield curve

In [ ]:
curve = nelson_siegel.us_curve()
curve.summary()
print(f'10Y yield: {curve.yield_at(10):.3%}')
print(f'10Y-2Y spread: {curve.yield_at(10)-curve.yield_at(2):+.3%}')

## 10. Audit + earnings quality

In [ ]:
from finverse import audit
audit(model).summary()
earnings_quality.score(data).summary()
benford.test_financials(data).summary()